# Caching for LLM Applications

Every LLM call costs money and time, and production traffic is repetitive: the same FAQ-shaped questions, the same embeddings, the same tool lookups, the same system prompt resent on every agent-loop iteration. Caching is usually the highest-leverage optimization an AI engineer can make, and this notebook builds up the three places a cache can live:

```text
your application -> response caches   (exact-match, then semantic)
your application -> artifact caches   (embeddings, tool results)
the provider     -> prompt cache      (server-side prefix reuse)
```

Along the way you will meet the one place caching goes badly wrong — serving a cached answer to a question that only *looks* the same — and it goes wrong worst in exactly our domain.

> This is an educational cat health assistant, not a veterinary care tool. That matters for caching too: a cache must never make an emergency answer stale or serve one user's context to another.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Measure the latency and token cost of repeated LLM calls.
- Add an exact-match response cache and explain its limits.
- Build a semantic cache with embeddings and a similarity threshold.
- Explain why semantic caching is dangerous in high-stakes domains and how to bound the risk.
- Cache embeddings and tool results inside an agent.
- Structure prompts so provider-side prompt caching actually fires.

## Table of Contents

- Task 1: Environment Setup
- Task 2: The Cost of Asking Twice
- Task 3: Exact-Match Response Caching
- Task 4: Semantic Caching From Scratch
- Task 5: Caching Embeddings and Tool Results
- Task 6: Provider-Side Prompt Caching

## Task 1: Environment Setup

From the `12_Production_Agent_Patterns` folder, install dependencies with uv (skip if you already did this for another notebook in this session):

```bash
uv sync
```

### Imports

This notebook uses:

- `set_llm_cache` and `InMemoryCache` for LangChain's built-in exact-match cache
- `OpenAIEmbeddings` and numpy for the from-scratch semantic cache
- The raw `openai` client for Task 6, because we need the usage details LangChain abstracts away

In [1]:
from getpass import getpass
import hashlib
import os
import time

import numpy as np
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from openai import OpenAI

### API Keys and Models

Same setup as Notebook 1, plus the embedding model that powers the semantic cache.

In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

os.environ.setdefault("LANGSMITH_PROJECT", "aim-session-12-caching")

chat_model_name = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
embedding_model_name = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")

llm = ChatOpenAI(model=chat_model_name)
embeddings = OpenAIEmbeddings(model=embedding_model_name)

print(f"Chat model: {chat_model_name}")
print(f"Embedding model: {embedding_model_name}")

Chat model: gpt-5.4-mini
Embedding model: text-embedding-3-small


## Task 2: The Cost of Asking Twice

First, establish the baseline we are trying to beat. A tiny timing helper, then the same question asked twice — full price both times.

In [3]:
def timed(fn, *args, **kwargs):
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - start


QUESTION = "How often should I feed an adult indoor cat? Answer in two sentences."

reply, seconds = timed(llm.invoke, QUESTION)
usage = reply.usage_metadata
print(f"first ask:  {seconds:.2f}s | {usage['input_tokens']} tokens in, {usage['output_tokens']} tokens out")
print(f"answer: {reply.content}\n")

reply_again, seconds_again = timed(llm.invoke, QUESTION)
usage_again = reply_again.usage_metadata
print(f"second ask: {seconds_again:.2f}s | {usage_again['input_tokens']} tokens in, {usage_again['output_tokens']} tokens out")

first ask:  3.08s | 21 tokens in, 55 tokens out
answer: Most adult indoor cats do well with **2 meals per day**, about 8–12 hours apart. If your cat is free-fed, overweight, underweight, or has a medical condition, it’s best to ask your vet for a tailored feeding schedule.

second ask: 1.53s | 21 tokens in, 62 tokens out


Identical question, identical cost, seconds of latency — and possibly a *different* answer, since nothing ties the two calls together. Multiply by every user who asks some version of "how often should I feed my cat" and the waste is obvious. Everything below is about not paying twice.

## Task 3: Exact-Match Response Caching

LangChain has a global LLM cache: one call to `set_llm_cache` and every subsequent `.invoke()` is keyed by the exact prompt string plus the model's parameters. A hit skips the network entirely — zero tokens, near-zero latency, and (a subtle bonus) a *deterministic* repeated answer.

The same experiment, with the cache on:

In [4]:
set_llm_cache(InMemoryCache())

_, cold_seconds = timed(llm.invoke, QUESTION)
cached_reply, warm_seconds = timed(llm.invoke, QUESTION)

print(f"cold (fills cache): {cold_seconds:.2f}s")
print(f"warm (cache hit):   {warm_seconds:.4f}s")

# Change ONE character -- a trailing space -- and it is a different key.
_, near_miss_seconds = timed(llm.invoke, QUESTION + " ")
print(f"one character off:  {near_miss_seconds:.2f}s -> full price again")

cold (fills cache): 1.33s
warm (cache hit):   0.0006s
one character off:  1.55s -> full price again


That last line is the whole limitation. Users never phrase a question identically twice, so an exact-match cache mostly helps with *programmatic* repetition — retries, batch jobs, agents re-asking a fixed sub-question, and (as in the evaluation sessions) re-running notebooks without re-billing every cell. For human traffic, we need a cache that understands that two phrasings can be the same question.

## Task 4: Semantic Caching From Scratch

A semantic cache stores `(query, embedding, answer)` triples. On lookup, embed the new query and compare it to stored queries with cosine similarity — the same operation as dense retrieval in Session 1, pointed at your own past answers instead of documents. If the best match clears a threshold, serve the stored answer for the price of one embedding call (roughly a hundredth of a cent) instead of a chat completion.

The threshold is the entire design. Too low and different questions collide; too high and paraphrases miss. Build it, then break it.

In [5]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


class SemanticCache:
    """Serve a stored answer when a new query is close enough to an old one."""

    def __init__(self, embed, threshold: float = 0.90):
        self.embed = embed  # callable: str -> list[float]
        self.threshold = threshold
        self.entries: list[tuple[str, np.ndarray, str]] = []

    def lookup(self, query: str) -> tuple[str | None, float, str | None]:
        """Return (answer or None, best_score, best_matching_query)."""
        if not self.entries:
            return None, 0.0, None
        vector = np.array(self.embed(query))
        scores = [
            (cosine(vector, stored_vector), stored_query, answer)
            for stored_query, stored_vector, answer in self.entries
        ]
        best_score, best_query, best_answer = max(scores, key=lambda s: s[0])
        if best_score >= self.threshold:
            return best_answer, best_score, best_query
        return None, best_score, best_query

    def store(self, query: str, answer: str) -> None:
        self.entries.append((query, np.array(self.embed(query)), answer))


semantic_cache = SemanticCache(embed=embeddings.embed_query, threshold=0.90)


def cached_ask(question: str) -> str:
    answer, score, matched_query = semantic_cache.lookup(question)
    if answer is not None:
        print(f"[HIT  {score:.3f}] matched stored query: {matched_query!r}")
        return answer
    print(f"[MISS {score:.3f} vs stored: {matched_query!r}]")
    answer = str(llm.invoke(question).content)
    semantic_cache.store(question, answer)
    return answer

In [6]:
answer_one = cached_ask("How often should I feed an adult indoor cat?")
print(answer_one, "\n")

# A paraphrase the exact-match cache would have missed:
answer_two = cached_ask("What's the right feeding schedule for a grown indoor cat?")
print(answer_two)

[MISS 0.000 vs stored: None]
Most adult indoor cats do well with **2 meals per day** — once in the **morning** and once in the **evening**.

A few notes:
- **Kittens** need more frequent meals.
- Some adult cats can do fine with **free-feeding** (food left out all day), but many indoor cats overeat that way.
- The right amount depends on your cat’s **weight, age, activity level, and the food’s calorie content**.

If you want, I can also help you figure out **how much** to feed your cat per day. 

[MISS 0.826 vs stored: 'How often should I feed an adult indoor cat?']
A good feeding schedule for a grown indoor cat is usually:

- **2 meals a day** — one in the morning and one in the evening
- **Same times each day** to keep routine and help with digestion and behavior
- **Portion-controlled** so you match the cat’s age, weight, and activity level

### Typical approach
- **Adult cats:** 2 meals/day is most common
- **Very food-driven cats or cats prone to vomiting from an empty stomach:** 

The paraphrase hits, and the second answer costs one embedding call. Now the failure mode. These two questions differ by one word:

In [7]:
chocolate = "My cat just ate chocolate. What should I do?"
chicken = "My cat just ate chicken. What should I do?"

vec_chocolate = np.array(embeddings.embed_query(chocolate))
vec_chicken = np.array(embeddings.embed_query(chicken))

print(f"similarity(chocolate, chicken) = {cosine(vec_chocolate, vec_chicken):.3f}")

similarity(chocolate, chicken) = 0.743


One of those questions is about dinner; the other is a poisoning emergency. If that similarity clears your threshold, a user reporting chocolate ingestion gets a cheerful cached answer about chicken being a fine treat — the worst possible output, served *faster and with more confidence* than a live model would have.

Embeddings measure how alike sentences look, not how much the difference matters. In cat health, one token can be the difference between "fine" and "call poison control". Two consequences for real systems:

- **The threshold cannot save you.** Whatever value you pick, some pair of critically different queries sits just above it.
- **High-stakes queries must bypass the cache entirely.** Notebook 1's deterministic emergency rail is exactly the right gate: if `run_input_rails` says `escalate`, never consult (and never populate) the semantic cache. Guardrails and caches are not separate features — the rails decide what is *safe to cache*.

A production semantic cache adds more on top of the bypass: TTL expiry, a bounded size with eviction, and per-user scoping so one user's cached answer never reaches another.

## Task 5: Caching Embeddings and Tool Results

Responses are not the only thing worth caching — anything deterministic and repeated is a candidate. Two artifacts dominate in agent systems:

**Embeddings.** The semantic cache above re-embeds every stored query it compares against — wasteful, since embedding a fixed string is deterministic. (Notice `SemanticCache.store` already avoids this by embedding once at insert; the lookup-side query embedding is the only live call.) The same trick, keyed by a content hash, speeds up re-indexing pipelines where most documents have not changed:

In [8]:
class CachingEmbedder:
    """Wrap an embed function with a content-hash keyed cache."""

    def __init__(self, embed):
        self.embed = embed
        self.store: dict[str, list[float]] = {}
        self.hits = 0
        self.misses = 0

    def __call__(self, text: str) -> list[float]:
        key = hashlib.sha256(text.encode()).hexdigest()
        if key in self.store:
            self.hits += 1
        else:
            self.misses += 1
            self.store[key] = self.embed(text)
        return self.store[key]


caching_embedder = CachingEmbedder(embeddings.embed_query)

_, cold_seconds = timed(caching_embedder, "grooming a long-haired cat")
_, warm_seconds = timed(caching_embedder, "grooming a long-haired cat")
print(f"cold: {cold_seconds:.3f}s   warm: {warm_seconds * 1000:.3f}ms")
print(f"{caching_embedder.hits} hit / {caching_embedder.misses} miss")

cold: 0.335s   warm: 0.016ms
1 hit / 1 miss


**Tool results.** When an agent's tool hits a slow or expensive backend — a database, a third-party API, a long document fetch — cache inside the tool with a TTL (time-to-live), so repeated agent loops and repeated users do not repeat the work. The `time.sleep` below stands in for the slow backend:

In [9]:
TOOL_CACHE: dict[str, tuple[float, str]] = {}
TOOL_TTL_SECONDS = 300.0
slow_lookups = {"count": 0}

CARE_TOPICS = {
    "feeding": "Adult cats do well with two measured meals a day; kittens need three to four.",
    "grooming": "Brush short-haired cats weekly and long-haired cats daily.",
    "litter box": "One box per cat plus one extra, scooped daily.",
}


@tool
def care_guide_lookup(topic: str) -> str:
    """Look up a cat-care guide entry for a topic such as feeding, grooming, or litter box."""
    key = topic.lower().strip()
    now = time.monotonic()

    if (hit := TOOL_CACHE.get(key)) and now - hit[0] < TOOL_TTL_SECONDS:
        return hit[1]

    slow_lookups["count"] += 1
    time.sleep(1.5)  # stand-in for a slow database or third-party API
    result = next(
        (entry for name, entry in CARE_TOPICS.items() if name in key or key in name),
        f"No entry for {topic!r}. Topics: {', '.join(CARE_TOPICS)}.",
    )

    TOOL_CACHE[key] = (now, result)
    return result


caching_agent = create_agent(
    model=llm,
    tools=[care_guide_lookup],
    system_prompt=(
        "You are an educational cat health assistant. Use the care guide tool "
        "for care questions. Do not diagnose or prescribe; keep answers concise."
    ),
)

for attempt in (1, 2):
    _, seconds = timed(
        caching_agent.invoke,
        {"messages": [{"role": "user", "content": "How should I set up litter boxes for two cats?"}]},
    )
    print(f"run {attempt}: {seconds:.2f}s total, slow lookups so far: {slow_lookups['count']}")

run 1: 4.66s total, slow lookups so far: 1
run 2: 1.70s total, slow lookups so far: 1


The second run still pays for its model calls, but the slow backend ran once. The TTL is the honesty knob: five minutes means "I accept serving five-minute-old care guidance", which is fine here — and would not be fine for, say, clinic appointment availability.

## Task 6: Provider-Side Prompt Caching

The third cache is not yours: the provider keeps the computed state of recent prompt *prefixes* and reuses it. OpenAI does this automatically for prompts longer than 1024 tokens, bills cached input tokens at a steep discount, and reports what it reused in the response's usage details. Anthropic exposes the same idea explicitly via `cache_control` markers on message blocks.

You do not call this cache — you *dress for it*. Only an identical prefix can be reused, so the rule is:

```text
stable content first  -> system prompt, tool definitions, few-shot examples
variable content last -> the user's question, retrieved context
```

Put a timestamp or user name at the top of your system prompt and you have disabled prompt caching for your whole application. Agents are the biggest beneficiary: every loop iteration and every user resends the same system-prompt-plus-tools prefix.

We drop to the raw OpenAI client here because we need `usage.prompt_tokens_details`:

In [10]:
client = OpenAI()

CARE_RULES = [
    "Provide educational cat care information only, never diagnosis or dosage.",
    "Encourage a veterinarian visit for anything medical or behavioral changes.",
    "Be specific about life stage: kitten, adult, and senior needs differ.",
    "Prefer positive-reinforcement guidance for behavior questions.",
]

# Repeat the rules to push the stable prefix well past the 1024-token minimum.
care_manual = "You are an educational cat health assistant.\n" + "\n".join(
    f"Rule {i}: {rule}" for i, rule in enumerate(CARE_RULES * 60, start=1)
)


def ask_with_manual(question: str) -> None:
    response = client.chat.completions.create(
        model=chat_model_name,
        messages=[
            {"role": "system", "content": care_manual},  # stable prefix first
            {"role": "user", "content": question},       # variable suffix last
        ],
    )
    usage = response.usage
    cached_tokens = usage.prompt_tokens_details.cached_tokens if usage.prompt_tokens_details else 0
    print(f"prompt tokens: {usage.prompt_tokens:>5} | served from prompt cache: {cached_tokens:>5}")


ask_with_manual("How often should I trim my cat's claws?")
ask_with_manual("Is wet or dry food better for a senior cat?")

prompt tokens:  3868 | served from prompt cache:     0
prompt tokens:  3869 | served from prompt cache:  3328


The second call should show most of the prompt served from cache — different question, identical prefix. If `cached tokens` reads 0 on the second call, wait a few seconds and re-run it: provider caches take a moment to register a new prefix, and entries expire after minutes of inactivity, so this is an optimization you *measure*, never one you assume.

The three caches compose. A request that misses the semantic cache still hits the prompt cache for its prefix; a tool call that misses the tool cache still benefits from cached embeddings. Each layer removes a different kind of repeated work.

## Summary

You built the three layers of caching that show up in nearly every production LLM system:

- **Exact-match response caching**: free wins on programmatic repetition, defeated by a single changed character.
- **Semantic caching**: catches paraphrases for the price of an embedding, but similarity is not equivalence — high-stakes queries must bypass it, and guardrails are what decide which queries those are.
- **Artifact caching**: embeddings keyed by content hash; tool results with a TTL that states, honestly, how stale you are willing to be.
- **Provider-side prompt caching**: not called but dressed for — stable prefix first, variable content last, savings measured in the usage details rather than assumed.

### Further Reading

- [OpenAI prompt caching guide](https://platform.openai.com/docs/guides/prompt-caching)
- [Anthropic prompt caching documentation](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)
- [GPTCache: semantic caching for LLM apps](https://github.com/zilliztech/GPTCache)
- [LangChain model caches](https://python.langchain.com/docs/integrations/llm_caching/)

### Notebook Output Guidance

Keep the cold/warm timing comparisons, the semantic HIT/MISS lines, the chocolate/chicken similarity score, and the prompt-cache token counts — those numbers are the argument this notebook makes. Timings vary by network and provider load; that is expected and worth a sentence in your notes rather than a re-run chase.